### Preprocess data

In [ ]:

import scanpy as sc
import numpy as np
import pandas as pd

import argparse
import seaborn as sns
import sys
import logging
import configparser
import anndata
import matplotlib.pyplot as plt
import re
import matplotlib.patches as mpatches
import os
import argparse
import copy
import h5py

from pathlib import Path # For easier path handling
import anndata as ad

import copy
import scipy.sparse as sp

# Configure logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

# Read and filter the Hypr-seq anndata
def filter_hypr(adata, probe_level=False):
    # Remove cells with less than 1000 UMIs per cell
    sc.pp.filter_cells(adata, min_counts=1000)
    print("\n--- Performing renaming ---")
    
    # merge the probe name, making the cell by probe matrix to cell by gene matrix
    # Splitting at the underscore to separate gene names from probe numbers
    if not probe_level:
        gene_names = ["_".join(name.split('_')[:-1]) for name in adata.var_names]
    else:
        gene_names = [name for name in adata.var_names]

    # Add the gene names as a new column in the DataFrame of the .var slot
    adata.var['gene_name'] = gene_names
    
    adata.X = adata.X.toarray()
    # Convert the AnnData to a DataFrame for easier manipulation
    adata_df = pd.DataFrame(adata.X.T, index=adata.var_names, columns=adata.obs_names)

    # Use the gene names to sum the counts
    # Group by the new gene names and sum across columns (probes for the same gene)
    aggregated_data = adata_df.groupby(adata.var['gene_name']).sum()

    # Transpose back to original shape (samples as rows, genes as columns)
    aggregated_data = aggregated_data.T

    # Create new AnnData object with the aggregated data
    adata_aggregated = sc.AnnData(X=aggregated_data)
    # Copy the metadata from the original AnnData object
    adata_aggregated.obs = adata.obs.copy()
    # Optionally, copy over any relevant .uns data (unsupervised annotations, such as PCA, neighbors, etc.)
    #adata_aggregated.uns = adata.uns.copy()

    return adata_aggregated




# function reads the loomfile downloaded from Tapestri portal
def read_tapestri_H5(filename):
    """
    Read data from MissionBio's formatted loom file.

    Parameters
    ----------
    filename : str
        Path to the loom file (.loom)

    Returns
    -------
    anndata.AnnData
        An anndata object with the following layers:
        adata.X: GATK calls
        adata.layers['e']: reads with evidence of mutation
        adata.layers['no_e']: reads without evidence of mutation
    """

    with h5py.File(filename, "r") as file:
        # get the variant name, amplicon, chromosome, location]
        # import pdb; pdb.set_trace()
        variant_names = file['assays']["dna_variants"]["ca"]['id'][:] # binary
        variant_names = np.array([i.decode("utf-8") for i in variant_names])


        amplicon_names = file['assays']['dna_variants']['ca']['amplicon'][:] # binary
        amplicon_names = np.array([i.decode("utf-8") for i in amplicon_names])

        chromosome = file['assays']['dna_variants']['ca']['CHROM'][:] # binary
        chromosome = np.array([i.decode("utf-8") for i in chromosome])


        location = file['assays']['dna_variants']['ca']['POS'][:]

        # get the barcode
        barcodes = file['assays']['dna_variants']['ra']['barcode'][:] # binary
        barcodes = np.array([i.decode("utf-8") for i in barcodes])

        mutation_matrix = file['assays']['dna_variants']['layers']['NGT'][:,:]

        adata = anndata.AnnData(X=mutation_matrix, dtype=np.int8)
        adata.obs_names = barcodes
        adata.var_names = variant_names
        adata.varm["amplicon"] = amplicon_names
        adata.varm["chrom"] = chromosome
        adata.varm["loc"] = location
    file.close()

    return adata



# Function for finding the intersecting barcodes between the modalities
# We can drop the idea of mudata for simplity
def find_intersecting_and_filter(adata_hypr, adata_h5):
    """
    Find the intersecting barcode, reorder, and return new AnnData objects for both datasets.
    """
    # Find intersecting barcodes
    obs_1, obs_2 = adata_hypr.obs_names, adata_h5.obs_names
    cmn_barcodes, idx_1, idx_2 = np.intersect1d(obs_1, obs_2, return_indices=True)

    # Logging the number of barcodes
    logger.info(f"Found {len(obs_1)} barcodes in modality hypr seq")
    logger.info(f"Found {len(obs_2)} barcodes in modality loom file")
    logger.info(f"Found {len(idx_1)} intersecting barcodes")

    # Subset and reorder both datasets based on the intersecting indices
    adata_hypr_new = adata_hypr[idx_1, :]
    adata_h5_new = adata_h5[idx_2, :]

    # # Ensure the order of barcodes is the same in both datasets
    # adata_hypr_new = adata_hypr_new[cmn_barcodes, :]
    # adata_h5_new = adata_h5_new[cmn_barcodes, :]

    return adata_hypr_new, adata_h5_new
    

# Step 1, determine the germline mutation from the loom file
# We envision that if certain mutation appeared too many times in the dataset
# (e.g., more than 25% cells have the homo mutation, we call it homo germline)
def call_germline_mutations(adata_h5, cutoff=0.25):

    fraction_het = np.mean(adata_h5.X == 1, axis=0)
    het_germline = adata_h5.var_names[fraction_het > cutoff]

    fraction_hom = np.mean(adata_h5.X == 2, axis=0)
    hom_germline = adata_h5.var_names[fraction_hom > cutoff]

    # return {
    #     "het_germline": list(het_germline),
    #     "hom_germline": list(hom_germline)
    #     }
    return list(het_germline) + list(hom_germline)
        




# For AAV control, we may ignore mixed cells and bystander effect
def call_AAV_control_edits(adata_h5, config_path='IRF4.ini', control_name="AAVS-ACBE3"):
    """
    Input: Tapestri loom file and mutation configuration 
    Current we only accept 1 type of AAV control (should be a continuous region)
    Output: All the AAV edits found in the loom file. 

    This function will not modify adata_h5
    """
    # Read configuration settings
    config = configparser.ConfigParser()
    config.read(config_path)
    
    chrom_name = config.get(control_name, 'chrom_name')
    start = config.getint(control_name, 'start')
    end = config.getint(control_name, 'end')
    variant_alleles = config.get(control_name, 'variant_alleles')
    
    # Parsing variant names
    variant_names = np.asarray(adata_h5.var_names.values)
    chrom = [name.split(':')[0] for name in variant_names]
    # import pdb; pdb.set_trace()
    loc = [int(name.split(':')[1]) for name in variant_names]
    edit_type = [name.split(':')[2] for name in variant_names]

    control_editing = []
    complement_rule = {'C': 'G', 'G': 'C', 'A': 'T', 'T': 'A'}
    try:
        reverse_variant_alleles = complement_rule[variant_alleles[0]] + "/" + complement_rule[variant_alleles[-1]]
    except:
        raise ValueError(f"The variant alleles is not supported. Current variant alleles is {variant_alleles}")
    # Identify control editing cells
    for i in range(len(chrom)):
        if chrom[i] == chrom_name and start <= loc[i] < end+1:
            if edit_type[i] in [variant_alleles, reverse_variant_alleles]:
                control_editing.append(variant_names[i])

    return control_editing


def call_AAV_hom_pure_cells(
    adata_hypr,
    adata_h5,
    config_path,
    control_name,
    germline_amplicons,
    plot_path,
    window_size=10
):
    """
    Identifies cells that are "homozygous pure" for a specified AAV control locus.

    This function first finds all variants associated with the AAV control region
    and then, for each variant, identifies cells that are homozygous (genotype 2)
    and have no other bystander edits within the specified window.

    Args:
        adata_hypr (anndata.AnnData): The scRNA-seq AnnData object.
        adata_h5 (anndata.AnnData): The DNA variant AnnData object.
        config_path (str): Path to the .ini configuration file.
        control_name (str): The section name for the AAV control in the config file (e.g., "AAVS-ACBE3").
        germline_amplicons (list): List of germline mutations to exclude from bystander analysis.
        plot_path (str): Path to save diagnostic plots from the bystander analysis.
        window_size (int): The genomic window size to check for bystander edits.

    Returns:
        np.ndarray: An array of integer indices corresponding to the AAV "homozygous pure"
                    cells in adata_hypr.
    """
    logger.info(f"Identifying 'homozygous pure' cells for AAV control: {control_name}")

    # Step 1: Find all variant names that correspond to the AAV control locus
    # This logic is borrowed from your original `call_AAV_control_edits` function.
    config = configparser.ConfigParser()
    config.read(config_path)

    chrom_name = config.get(control_name, 'chrom_name')
    start = config.getint(control_name, 'start')
    end = config.getint(control_name, 'end')
    variant_alleles = config.get(control_name, 'variant_alleles')

    # Find potential reverse complement
    complement_rule = {'C': 'G', 'G': 'C', 'A': 'T', 'T': 'A'}
    reverse_variant_alleles = complement_rule[variant_alleles[0]] + "/" + complement_rule[variant_alleles[-1]]

    # Find all variants in the H5 file that match the control criteria
    aav_control_variants = []
    for var in adata_h5.var_names:
        v_chrom, v_loc_str, v_edit = var.split(':')
        v_loc = int(v_loc_str)
        if v_chrom == chrom_name and start <= v_loc <= end and v_edit in [variant_alleles, reverse_variant_alleles]:
            aav_control_variants.append(var)

    if not aav_control_variants:
        logger.warning(f"No AAV control variants found for {control_name}. Returning zero cells.")
        return np.array([], dtype=int)

    logger.info(f"Found {len(aav_control_variants)} AAV variants to check: {aav_control_variants}")

    # Step 2: For each AAV variant, find the 'homozygous pure' cells.
    all_hom_pure_barcodes = []
    for target_loc in aav_control_variants:
        # We can reuse the existing function designed for SNPs to do the classification!
        _, _, _, hom_pure_barcodes = call_single_loci_cells(
            target_loc=target_loc,
            adata_h5=adata_h5,
            window_size=window_size,
            germline_amplicons=germline_amplicons,
            snp_name=f"{control_name}_{target_loc.replace(':', '_')}", # Create a unique name for plots
            plot_path=plot_path
        )
        if hom_pure_barcodes:
            all_hom_pure_barcodes.extend(hom_pure_barcodes)

    if not all_hom_pure_barcodes:
        logger.warning(f"Found AAV edits, but none were classified as 'homozygous pure' for {control_name}.")
        return np.array([], dtype=int)

    # Step 3: Get the unique list of cell barcodes and find their indices in adata_hypr
    unique_barcodes = np.unique(all_hom_pure_barcodes)
    _, aav_cell_indices, _ = np.intersect1d(adata_hypr.obs_names, unique_barcodes, return_indices=True)

    logger.info(f"Identified {len(aav_cell_indices)} unique 'homozygous pure' AAV control cells.")
    return aav_cell_indices



def call_AAV_cells(adata_hypr, adata_h5, AAV_editing):
    """
    Annotate the hypr adata (scRNA-seq data) by the AAV_edit found in call_AAV_edits.
    We do not perform annotation in the function to ensure consistency
    """
    all_idx = []
    for edit in AAV_editing:
        # get the idx of the edit we found
        idx = adata_h5.var_names.get_loc(edit)
        # if the value is 1/2 (hete/homo), we annotate the cell as control edit cell
        het_idx, hom_idx = [np.flatnonzero(adata_h5.X[:, idx] == i) for i in (1, 2)]
        AAV_edit_idx = np.concatenate([het_idx, hom_idx])
        AAV_cells_barcode_loom = adata_h5.obs_names[AAV_edit_idx]
        # find the AAV cells index in that of the adata_hypr
        _, idx1, _ = np.intersect1d(adata_hypr.obs_names, AAV_cells_barcode_loom, return_indices=True)
        all_idx.append(idx1)
        
    return np.unique(np.concatenate(all_idx))



def get_nearby_variants(
    variant_names, # List of all variant in the loom file
    target_loc, # loci of interest
    germline_amplicons, # List of Germline mutations 
    window_size # window size
):
    """
        Get nearby variants within a window size of the target_loc.
        Args:
            variant_names (list): List of variant names.
            target_loc (str): Target loci (e.g. chr1:123456:A/G).
            germline_amplicons (list): List of germline amplicons.
            window_size (int): Window size.
        Returns:
            list: List of nearby variants.
    """
    chrom_t, loc_t, _ = target_loc.split(":")
    nearby_variants = []
    for variant in variant_names:
        chrom, loc, edit_type = variant.split(":")
        is_valid_edit_type = True
        is_not_germline = variant not in germline_amplicons
        if (
            chrom == chrom_t
            and (int(loc_t) - window_size) <= int(loc) <= (int(loc_t) + window_size)
            and is_valid_edit_type
            and is_not_germline
        ):
            nearby_variants.append(variant)

    return nearby_variants



def plot_mutant_types(target_loc, new_matrix, mt_type, snp_name, save_path, index=""):
    """
    Plot the nearby genotype heatmap
    """
    mutation_counts = new_matrix.apply(np.count_nonzero, axis=0)
    plt.figure(figsize=(10, 6))
    bars = plt.bar(mutation_counts.index, mutation_counts.values)
    for i, idx in enumerate(mutation_counts.index):
        plt.text(
            bars[i].get_x() + bars[i].get_width() / 2,
            bars[i].get_height(),
            str(mutation_counts.values[i]),
            ha="center",
            va="bottom",
        )
    if target_loc in mutation_counts.index:
        bars[mutation_counts.index.tolist().index(target_loc)].set_color("r")
    red_patch = mpatches.Patch(color="red", label="Target Loci")
    plt.legend(handles=[red_patch])

    # we need to rename the target_loc to ensure we do not introduce extra path problems
    target_loc = target_loc.replace("/", "-")


    plot_title = (
        f"Mutation Counts: {mt_type} - {target_loc} ({index}) with total cells {len(new_matrix)}"
    )
    plt.title(plot_title)
    plt.xticks(rotation=90)
    plt.xlabel("")
    plt.ylabel("Number of Mutated Cells")

    save_index = f"{snp_name}_{mt_type}_mutant_counts_{index}.pdf"
    plt.savefig(os.path.join(save_path, save_index), bbox_inches="tight")
    plt.close()

    # Plot a cluster map of matrix but cluster rows only.
    sns.clustermap(
        new_matrix,
        cmap="YlGnBu",
        row_cluster=True,
        col_cluster=False,
        figsize=(10, 10),
    )
    plt.title(
        f"Mutation Matrix: {mt_type} - {snp_name} aka {target_loc} with shape {new_matrix.shape}"
    )
    plt.xlabel("Variant")
    plt.ylabel("Cell Barcode")
    save_index = f"{snp_name}_{mt_type}_mutant_matrix_{index}.pdf"
    plt.savefig(os.path.join(save_path, save_index), bbox_inches="tight")
    plt.close()




def process_bystander_cells(target_loc, cells, nearby_variants, adata_h5):
    """
    Identify which cells have bystander editing.
    Args:
        target_loc (str): Target loci (e.g. chr1:123456:A/G).
        cells (list): List of cells (that supposedly have the mutation).
        nearby_variant: adata that includes only the nearby cells
    Returns:
        list: List of pure cells.
        pd.DataFrame: New matrix.
        pd.DataFrame: Matrix with false amplicons.
    """
    nearby_adata = adata_h5[cells, nearby_variants].copy()
    new_matrix = pd.DataFrame(
        nearby_adata.X, index=nearby_adata.obs_names, columns=nearby_adata.var_names
    )
    # Change all 3 to 0 in the new matrix, but excluding the target loci column.
    for col in new_matrix.columns:
        if col != target_loc:
            new_matrix[col] = new_matrix[col].apply(
                lambda val: 0 if val == 3 else val
            )

    # Remove the target loci column to generate a bystander position only matrix.
    matrix_without_target_loci = new_matrix.loc[
        :, new_matrix.columns != target_loc
    ].copy()
    bystander_sum_for_every_cell = matrix_without_target_loci.sum(axis=1)
    # Remove cells that have a row sum of 0, or there's no bystander editing.
    matrix_without_target_loci = matrix_without_target_loci.loc[
        bystander_sum_for_every_cell != 0
    ]

    # If there's no bystander editing, return the original cells.
    if len(matrix_without_target_loci) == 0:
        # Set the bystander as the empty set
        return set(cells), set(), new_matrix

    # Everything that's left is a bystander cell.
    pure_cells = set(cells) - set(matrix_without_target_loci.index)
    return pure_cells, set(matrix_without_target_loci.index), new_matrix





def call_single_loci_cells(
    target_loc, # the loci of interest
    adata_h5, # the loom matrix to retrieve info
    window_size, # the window size 
    germline_amplicons, # The germline mutations
    snp_name, # the name of the loci
    plot_path, # the path to save the figures
):
    """
    Process a single target loci. 
    We will annotate all cells that include the loci
    The values represent the mutant type:
        0: Background
        1: Pure heterozygous
        2: Heterozygous with bystander editing
        3: Pure homozygous
        4: Homozygous with bystander editing
    """
    # import pdb; pdb.set_trace()
    # Find the idx of the loci
    idx = adata_h5.var_names.get_loc(target_loc)
    # Annotate cells that are background, het, and hom based on the loom matrix
    bkg_cells, het_cells, hom_cells = [
            adata_h5.obs_names[np.flatnonzero(adata_h5.X[:, idx] == i)] for i in (0, 1, 2)
            ]
    # Find all nearby variant of the give loci
    variant_names = adata_h5.var_names
    nearby_variants = get_nearby_variants(
            variant_names, target_loc, germline_amplicons, window_size
    )

    het_pure, het_bystander, het_matrix = process_bystander_cells(
            target_loc, het_cells, nearby_variants, adata_h5
    )
    hom_pure, hom_bystander, hom_matrix = process_bystander_cells(
            target_loc, hom_cells, nearby_variants, adata_h5
    )

    try:
        plot_mutant_types(target_loc, het_matrix, "Heterozygous", snp_name, plot_path, "1-0")
    except:
        # import pdb; pdb.set_trace()
        print(f"Cannot make plots for the {target_loc} for hete condition")
    try:
        plot_mutant_types(target_loc, hom_matrix, "Homozygous", snp_name, plot_path, "1-0")
    except:
        # import pdb; pdb.set_trace()
        print(f"Cannot make plots for the {target_loc} for the homo condition")

    return list(het_bystander), list(hom_bystander), list(het_pure), list(hom_pure)

    # adata_hypr.loc[hom_pure, "genotype"] = f"{target_loc}_homo_pure"
    # adata_hypr.loc[het_pure, "genotype"] = f"{target_loc}_hete_pure"


def call_wild_type_cells(
    target_loc,
    adata_h5,
    window_size,
    germline_amplicons
):
    """
    Identifies wild-type cells for a target locus and classifies them
    into 'pure' (no nearby edits) and 'bystander' (nearby edits present).

    This function is intended for specific loci where understanding the state of
    unedited cells is important.

    Args:
        target_loc (str): The genomic locus of interest (e.g., 'chr1:12345:A/G').
        adata_h5 (anndata.AnnData): The AnnData object with genotype information.
        window_size (int): The genomic window size to check for bystander edits.
        germline_amplicons (list): A list of germline variants to exclude from the bystander check.

    Returns:
        tuple: A tuple containing two lists of cell barcodes:
               - wt_pure_cells (list): Cells that are WT at the target and have no bystander edits.
               - wt_bystander_cells (list): Cells that are WT at the target but have bystander edits.
    """
    # Step 1: Find all cells that are wild-type (genotype 0) at the target locus
    try:
        idx = adata_h5.var_names.get_loc(target_loc)
    except KeyError:
        logger.warning(f"Target locus {target_loc} not found in adata_h5. Skipping.")
        return [], []

    wt_cells_mask = adata_h5.X[:, idx] == 0
    wt_cells = adata_h5.obs_names[wt_cells_mask].tolist()

    if not wt_cells:
        logger.info(f"No wild-type cells found for {target_loc}.")
        return [], []

    # Step 2: Get all relevant variants within the window, excluding the target itself
    variant_names = adata_h5.var_names
    nearby_variants = get_nearby_variants(
        variant_names, target_loc, germline_amplicons, window_size
    )
    
    # We only care about bystander edits, so the target locus is irrelevant for this check
    if target_loc in nearby_variants:
        nearby_variants.remove(target_loc)

    # If no other variants exist in the window, all WT cells are by definition "pure"
    if not nearby_variants:
        return wt_cells, []

    # Step 3: Check for the presence of bystander edits in the WT cell population
    # Create a sub-matrix of only WT cells and the nearby (bystander) variants
    nearby_adata = adata_h5[wt_cells, nearby_variants].copy()

    # Sum the genotype values across all nearby variants for each cell.
    # A sum > 0 indicates at least one bystander edit (het=1, hom=2).
    bystander_sum_per_cell = np.asarray(nearby_adata.X.sum(axis=1)).flatten()
    
    # Step 4: Classify cells based on the bystander sum
    # Create a temporary DataFrame for easy indexing
    wt_obs_df = pd.DataFrame(index=nearby_adata.obs_names)
    
    wt_pure_cells = wt_obs_df.index[bystander_sum_per_cell == 0].tolist()
    wt_bystander_cells = wt_obs_df.index[bystander_sum_per_cell > 0].tolist()

    return wt_pure_cells, wt_bystander_cells


def annotate_cells(adata):
    """
    Annotates each cell in adata.obs based on the SNP, control, and del columns, according to specified rules.
    
    Parameters:
    - adata: AnnData object containing the obs DataFrame with binary columns for each condition.
    
    Returns:
    - Annotations are stored in a new column 'annotation' in adata.obs.
    """
    
    # List of condition columns, based on the presence of SNPs, control, and del columns in adata.obs
    snp_columns = [col for col in adata.obs.columns if any(sub in col for sub in ["het_bystander", "het_pure", "hom_bystander", "hom_pure", "wt_bystander", "wt_pure"])]
    germline_cols = ['RORC-rs4845556_het_pure', 'RORC-rs4845556_het_bystander', 
    'RORC-rs11586691_het_bystander', 'RORC-rs11586691_het_pure',
    'RORC-rs7523001_het_pure', 'RORC-rs7523001_het_bystander']
    for col_name in germline_cols:
        try:
            snp_columns.remove(col_name)
        except:
            continue
    control_column = "AAVS_ACBE3"
    control_column_S4 = "AAVS4"

    # Define the function for annotating a single cell
    def annotate_row(row):
        # Check SNP-related rules
        snp_active = row[snp_columns].sum()  # Count number of active SNP columns (True values)
        
        if snp_active >= 2:  # More than one SNP column is True
            return "mixed"

        
        if row[control_column] and snp_active == 1:  # SNP and control active
            # Find which SNP column is active
            # snp_name = row[snp_columns][row[snp_columns] == True].index[0]
            return "mixed"
        
        if row[control_column_S4] and snp_active == 1:  # SNP and del active
            return "mixed"
        
        if row[control_column] and snp_active == 0:  # Only control active
            return "AAVS_ACBE3"
        
        if row[control_column_S4] and snp_active == 0:  # Only del active
            return "AAVS4"
        
        if snp_active == 1:  # Single SNP active, no control or del
            snp_name = row[snp_columns][row[snp_columns] == True].index[0]
            return snp_name
        
        # If no columns are active
        if snp_active == 0 and not row[control_column] and not row[control_column_S4]:
            return "unedited"
        
        # Default case, though we should not reach here
        return "unknown"
    # import pdb; pdb.set_trace()
    # Apply the annotation function to each row in a vectorized manner
    adata.obs['genotype_annotation'] = adata.obs.apply(annotate_row, axis=1)
    





def annotate_genotype(adata_h5, 
                    adata_hypr, 
                    config_path,
                    save_path="./",
                    plot_path="./"):
    """
    Annotate the adata_hypr, such that in the adata_hypr.obs, we have:
    Homo pure for SNP 1, 2, 3, …
    Homo bystander for SNP 1,2, 3, …
    Hete pure for SNP 1, 2, 3, …
    Hete bystander for SNP 1, 2, 3…
    Mixed cells
    AAVs: AAV_control, AAV_del
    Unedited. 
    """

    # Perform the annotation step. We need:
    # 1. Determine the germline mutations
    # 2. Annotate the AAV_control cells
    # 3. Annotate the AAV_del_control cells
    # 4. For each SNP we are interested in, find if it has bystander effect
    # 5. Determine the bystander mutations
    # 6.  Annotate all mixed cells. For example, for 2 SNPs we are interested in, 
    #  if they are co-edited, we need to count the number. We need to annotate it as mixed.
    

    germline_amplicons = call_germline_mutations(adata_h5, cutoff=0.25)

    # for the RORC data, we have 3 variant which are germline heterozygous
    if 'chr1:151772382:G/A' in germline_amplicons:
        germline_amplicons.remove('chr1:151772382:G/A')
    if 'chr1:151799191:T/C' in germline_amplicons:
        germline_amplicons.remove("chr1:151799191:T/C")
    if 'chr1:151803755:C/T' in germline_amplicons:
        germline_amplicons.remove("chr1:151803755:C/T")
    

    # Step 2, annotate the AAVS-ACBE3 base editing group cells
    # 2.1 Find the control editing locis 
    
    AAV_control_editing = call_AAV_control_edits(adata_h5, config_path=config_path, control_name="AAVS-ACBE3")
    # 2.2 Add a column to the adata_hypr obs and then perform the annotation only here
    adata_hypr.obs["AAVS_ACBE3"] = False
    AAV_control_cell_idx = call_AAV_cells(adata_hypr, adata_h5, AAV_editing=AAV_control_editing)
    adata_hypr.obs.iloc[AAV_control_cell_idx, adata_hypr.obs.columns.get_loc("AAVS_ACBE3")] = True

    # Next step, update the AAVS4 control as it's a double mutation
    AAV_S4_control_editing = call_AAV_control_edits(adata_h5, config_path=config_path, control_name="AAVS4")
    adata_hypr.obs["AAVS4"] = False
    AAV_S4_control_cell_idx = call_AAV_cells(adata_hypr, adata_h5, AAV_editing=AAV_S4_control_editing)
    adata_hypr.obs.iloc[AAV_S4_control_cell_idx, adata_hypr.obs.columns.get_loc("AAVS4")] = True

    AAV_control_cell_idx = call_AAV_hom_pure_cells(
        adata_hypr=adata_hypr,
        adata_h5=adata_h5,
        config_path=config_path,
        control_name="AAVS4",
        germline_amplicons=germline_amplicons,
        plot_path=plot_path
    )
    adata_hypr.obs["AAVS4_Hom_Pure"] = False
    adata_hypr.obs.iloc[AAV_control_cell_idx, adata_hypr.obs.columns.get_loc("AAVS4_Hom_Pure")] = True


    # import pdb; pdb.set_trace()
    # I need to perform 2 level of annotation. 
    # level 1: annotate the Base editing group
    # Step 4/5, Enumerate all other variants we are interested in
    # Initialize the config parser
    #-------------------------------------------------------------------------------------------------
    # Base Editing group ananotation
    #-------------------------------------------------------------------------------------------------
    config = configparser.ConfigParser()
    # Read the config file
    config.read(config_path)
    # Retrieve and return all section names
    sections = config.sections()
    exclude_sections = ["AAVS-ACBE3", "AAVS4"]


    SNPs = [
        section for section in config.sections()
        if config.has_option(section, 'group') 
        # and config.get(section, 'group') == 'Base Editing' 
        and section not in exclude_sections
    ]

    print(f"Found the following Base Editing SNPs to process: {SNPs}")


    SNPs_for_WT_analysis = ["RORC-rs4845556", "RORC-rs11586691", "RORC-rs7523001"]
    # Each snp is the section name we defined in the config file
    # Perform the annotation here
    annotation = ["het_bystander", "het_pure", "hom_bystander", "hom_pure"]
    # import pdb; pdb.set_trace()
    for snp in SNPs:
        chrom_name = config.get(snp, 'chrom_name')
        locus = config.getint(snp, 'locus')
        alleles = config.get(snp, 'variant_alleles')
        complement_rule = {'C': 'G', 'G': 'C', 'A': 'T', 'T': 'A'}
        try:
            complement_variant_alleles = complement_rule[alleles[0]] + "/" + complement_rule[alleles[-1]]
        except:
            raise ValueError(f"The variant alleles is not supported. Current variant alleles is {alleles}")
        # Transfer back to the original version
        target_loc = ":".join([chrom_name, str(locus), alleles])
        complement_target_loc = ":".join([chrom_name, str(locus), complement_variant_alleles])
        
        # Find which allele is the one we want, the original version or the complement one
        if target_loc in adata_h5.var_names:
            target_loc = target_loc
        elif complement_target_loc in adata_h5.var_names:
            target_loc = complement_target_loc
        else:
            print(f"Both the mutation {target_loc} and the reversed one {complement_target_loc} is not found in the loom file")
            continue


        # Get the index of the cells
        het_bystander, hom_bystander, het_pure, hom_pure = call_single_loci_cells(target_loc, adata_h5, window_size=10, 
            germline_amplicons=germline_amplicons, snp_name = snp, plot_path=plot_path)
        

        for i, idx_i in enumerate([het_bystander, het_pure, hom_bystander, hom_pure]):
            adata_hypr.obs[f"{snp}_{annotation[i]}"] = False
            adata_hypr.obs.loc[idx_i, f"{snp}_{annotation[i]}"] = True
        
        
        if snp in SNPs_for_WT_analysis:
            # import pdb; pdb.set_trace()
            print(f"Performing WT analysis for {snp}...")
            wt_pure, wt_bystander = call_wild_type_cells(
                target_loc, adata_h5, window_size=10,
                germline_amplicons=germline_amplicons
            )

            # Add the new WT annotations to adata_hypr.obs
            adata_hypr.obs[f"{snp}_wt_pure"] = False
            adata_hypr.obs.loc[wt_pure, f"{snp}_wt_pure"] = True
            
            adata_hypr.obs[f"{snp}_wt_bystander"] = False
            adata_hypr.obs.loc[wt_bystander, f"{snp}_wt_bystander"] = True

    # Step 7, for the cells that has genotypes, we need to ensure that they are not mixed cells
    annotate_cells(adata_hypr)

    return adata_hypr, adata_h5
    



def main():
    # Create the parser
    parser = argparse.ArgumentParser(description="Filter, Intersect, Merge, and Annotate genotype and phenotype data from multiple experiments.")

    # --- Define Input Datasets ---
    # This structure defines the paths and metadata for your 5 datasets.
    # MODIFY THIS SECTION with your actual file paths and desired batch names.
    # Assumes a base directory structure, adjust as needed.
    # You could also load this from a metadata CSV file.
    datasets = {
        'Acti_probe4': {
            'loom_path': '../Data/figure6/combined_Th17_Acti_4.dna.h5',
            'hypr_path': '../Data/figure6/acti_probe4.h5ad', # Or other hypr format
            'condition': 'Acti',
            'batch': 'probe4',
        },
        'Acti_probe5': {
            'loom_path': '../Data/figure6/combined_Th17_Acti_5.dna.h5',
            'hypr_path': '../Data/figure6/acti_probe5.h5ad',
            'condition': 'Acti',
            'batch': 'probe5',
        },
        'Acti_probe6': {
            'loom_path': '../Data/figure6/combined_Th17_Acti_6.dna.h5',
            'hypr_path': '../Data/figure6/acti_probe6.h5ad',
            'condition': 'Acti',
            'batch': 'probe6',
        },
        'PBS_probe2': {
            'loom_path': '../Data/figure6/combined_Th17_PBS_2.dna.h5',
            'hypr_path': '../Data/figure6/pbs_probe2.h5ad',
            'condition': 'PBS',
            'batch': 'probe2',
        },
        'PBS_probe3': {
            'loom_path': '../Data/figure6/combined_Th17_PBS_3.dna.h5',
            'hypr_path': '../Data/figure6/pbs_probe3.h5ad',
            'condition': 'PBS',
            'batch': 'probe3',
        },
    }

    # Add arguments for common paths and settings
    parser.add_argument('--config_path', type=str, required=True, help='The path to the variant configuration file (INI format)')
    parser.add_argument('--save_path', type=str, required=True, help='The base path to save annotated anndata files')
    parser.add_argument('--plot_path', type=str, required=True, help='The base path to save genotype clustermap and other figures')
    parser.add_argument('--probe_level', type=int, required=True, help="1 for probe level matrix, other numbers for gene_level")

    # Parse the arguments
    args = parser.parse_args()

    config_path = args.config_path
    save_path = Path(args.save_path) # Use pathlib
    plot_path = Path(args.plot_path) # Use pathlib
    probe_level = (args.probe_level == 1)

    # Create output directories if they don't exist
    save_path.mkdir(parents=True, exist_ok=True)
    plot_path.mkdir(parents=True, exist_ok=True)

    # --- Process and Filter Individual Datasets ---
    processed_hypr_list = []
    processed_loom_list = []

    print("--- Processing Individual Datasets ---")
    for j, (sample_name, meta) in enumerate(datasets.items()):
        print(f"\nProcessing sample: {sample_name}")
        loom_path = Path(meta['loom_path'])
        hypr_path = Path(meta['hypr_path'])

        if not loom_path.is_file() or not hypr_path.is_file():
            print(f"  Warning: Input file(s) not found for {sample_name}. Skipping.")
            continue

        try:
            # Read data
            adata_loom_orig = read_tapestri_H5(loom_path)
            adata_hypr_orig = sc.read(hypr_path)
            print(f"  Read loom: {adata_loom_orig.shape}")
            print(f"  Read hypr: {adata_hypr_orig.shape}")

            # Filter hypr
            adata_hypr_filtered = filter_hypr(adata_hypr_orig, probe_level=probe_level)
            print(f"  Filtered hypr: {adata_hypr_filtered.shape}")

            # Filter and intersect
            adata_hypr_intersect, adata_loom_intersect = find_intersecting_and_filter(adata_hypr_filtered, adata_loom_orig)


            if adata_hypr_intersect.shape[0] == 0 or adata_loom_intersect.shape[0] == 0:
                 print(f"  Skipping {sample_name} due to zero intersecting cells after filtering.")
                 continue

            print(f"  Intersected hypr: {adata_hypr_intersect.shape}")
            print(f"  Intersected loom: {adata_loom_intersect.shape}")

            # Add metadata BEFORE concatenation
            adata_hypr_intersect.obs['condition'] = meta['condition']
            adata_hypr_intersect.obs['batch'] = meta['batch']
            adata_hypr_intersect.obs['sample_name'] = sample_name # Keep original sample name

            adata_loom_intersect.obs['condition'] = meta['condition']
            adata_loom_intersect.obs['batch'] = meta['batch']
            adata_loom_intersect.obs['sample_name'] = sample_name # Keep original sample name

            # Append to lists for merging
            processed_hypr_list.append(adata_hypr_intersect)
            processed_loom_list.append(adata_loom_intersect)
            print(f"  Successfully processed and filtered {sample_name}.")

        except Exception as e:
            print(f"  Error processing sample {sample_name}: {e}")

    # --- Merge Datasets ---
    if not processed_hypr_list or not processed_loom_list:
        print("\nError: No datasets successfully processed. Exiting.")
        return

    print("\n--- Merging Processed Datasets ---")
    try:
        # Concatenate hypr AnnDatas
        adata_hypr_merged = ad.concat(
            processed_hypr_list,
            join='outer',  # Keep all genes/probes, fill missing with 0 or NaN
            label='sample_key', # Column name to store original list index (0, 1, 2...)
            index_unique='-', # Make cell barcodes unique (e.g., sample_name-barcode)
            merge='unique' # How to merge .var/.obs data (use 'unique' or 'same' if appropriate)
        )

        if isinstance(adata_hypr_merged.X, np.ndarray):
            print("adata_hypr_merged.X is a dense array. Filling NaNs with np.nan_to_num.")
            # Use np.nan_to_num to replace NaN with 0, modifying in-place for efficiency
            np.nan_to_num(adata_hypr_merged.X, nan=0, copy=False)

        # there are missing genes in PBS_probe2, we need to fill them with 0
        
        # Optional: Refill sample_name based on prefix if index_unique was used effectively
        # adata_hypr_merged.obs['sample_name'] = [idx.split('-')[0] for idx in adata_hypr_merged.obs_names]
        print(f"Merged hypr data shape: {adata_hypr_merged.shape}")
        print(f"Merged hypr obs columns: {adata_hypr_merged.obs.columns.tolist()}")

        # Concatenate loom AnnDatas
        adata_loom_merged = ad.concat(
            processed_loom_list,
            join='outer', # Keep all mutations/sites
            label='sample_key',
            index_unique='-',
            merge='unique'
        )
        if isinstance(adata_loom_merged.X, np.ndarray):
            print("adata_loom_merged.X is a dense array. Filling NaNs with np.nan_to_num.")
            np.nan_to_num(adata_loom_merged.X, nan=0, copy=False)
            
        # Ensure loom data is aligned to merged hypr data
        adata_loom_merged = adata_loom_merged[adata_hypr_merged.obs_names, :].copy()
        print(f"Merged loom data shape: {adata_loom_merged.shape}")
        print(f"Merged loom obs columns: {adata_loom_merged.obs.columns.tolist()}")

        # Check alignment
        if not all(adata_hypr_merged.obs_names == adata_loom_merged.obs_names):
             raise ValueError("Merged loom and hypr AnnData objects are not perfectly aligned by cell barcode.")

    except Exception as e:
        print(f"Error during AnnData concatenation: {e}")
        return

    # --- Annotate Merged Data ---
    print("\n--- Annotating Merged Data ---")
    try:
        # Annotate the merged data using the genotype info from merged loom
        # The annotate_genotype function needs to handle merged data correctly
        adata_hypr_annotated, adata_loom_annotated = annotate_genotype(
            adata_h5=adata_loom_merged,
            adata_hypr=adata_hypr_merged,
            config_path=config_path,
            save_path=str(save_path), # Pass as string
            plot_path=str(plot_path)  # Pass as string
        )
    except Exception as e:
        print(f"Error during annotation: {e}")
        return

    # --- Save Final Annotated Data ---
    print("\n--- Saving Final Annotated Data ---")
    try:
        if probe_level:
            hypr_name = save_path / "Annotated_phenotype_hypr_seq_probe_MERGED.h5ad"
        else:
            hypr_name = save_path / "Annotated_phenotype_hypr_seq_MERGED.h5ad"
        loom_name = save_path / "Annotated_genotype_loom_MERGED.h5ad"

        print(f"Saving annotated hypr data to: {hypr_name}")
        sc.write(hypr_name, adata_hypr_annotated)

        adata_loom_annotated.X = sp.csr_matrix(adata_loom_annotated.X)  # Ensure sparse matrix format
        print(f"Saving annotated loom data to: {loom_name}")
        sc.write(loom_name, adata_loom_annotated)

        print("Successfully saved final annotated files.")
    except Exception as e:
        print(f"Error saving final files: {e}")


if __name__ == "__main__":
    # config: rorc.ini
    main() 